In [ ]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.9 MB/s eta 0:00:00


In [ ]:
!pip install bertopic

import pandas as pd
import re

from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.9 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [ ]:
databert = pd.read_excel("/content/sentimen tweet_kosmetik.xlsx")
databert = databert.dropna(subset=["Text"])
databert

,Text,Tanggal,Bulan,Tahun,sentimen
0,enggak usah macem-macem lah kalau enggak paham...,16,11,2025,positif
1,Biar gada lagi oknum2 bikin kosmetik skincare ...,12,11,2025,netral
2,Slalu update product makeup/skincare dari BPOM...,6,11,2025,netral
3,untung deh gw belum beli skincare &amp; make u...,5,11,2025,netral
4,Sebelum beli kosmetik pastikan Pilih kosmetik ...,28,10,2025,netral
...,...,...,...,...,...
567,ya ampun korban pink flash ngeri bgt untung ga...,7,11,2025,netral
568,gila banyak bgt korbannya pink flash,7,11,2025,netral
569,@strsshdp @ywonnielvs Pink flash lagi di tarik...,7,11,2025,netral
570,Pink flash blm BPOM jifrrr fak,7,11,2025,netral


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"@\S+", " ", text)
    text = re.sub(r"#\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

databert["clean_text"] = databert["Text"].apply(clean_text)

In [ ]:
kata = {
    "g": "tidak",
    "ga": "tidak",
    "gak": "tidak",
    "nggak": "tidak",
    "ngga": "tidak",
    "enggak": "tidak",
    "engga": "tidak",
    "gk": "tidak",
    "gx": "tidak",
    "tdk": "tidak",
    "tak": "tidak",
    "kagak": "tidak",
    "ora": "tidak",
    "ndak": "tidak",

    "jgn": "jangan",
    "jngn": "jangan",

    "gw": "saya",
    "gue": "saya",
    "gua": "saya",
    "aq": "saya",
    "aku": "saya",
    "sy": "saya",
    "sya": "saya",

    "lu": "kamu",
    "loe": "kamu",
    "lo": "kamu",
    "elu": "kamu",
    "km": "kamu",
    "kmu": "kamu",

    "kak": "kakak",
    "kaka": "kakak",
    "sis": "kakak",
    "bro": "kakak",
    "gan": "kakak",

    "min": "admin",

    "guys": "teman",
    "guysss": "teman",
    "guysnya": "teman",
    "guysssnya": "teman",

    "yg": "yang",
    "yg.": "yang",
    "yng": "yang",

    "dgn": "dengan",
    "dg": "dengan",
    "dngn": "dengan",

    "dr": "dari",
    "dri": "dari",

    "utk": "untuk",
    "bwt": "buat",
    "buat": "untuk",
    "buatnya": "untuk",
    "u/": "untuk",
    "u": "untuk",

    "krn": "karena",
    "karna": "karena",
    "krna": "karena",
    "soalnya": "karena",

    "klo": "kalau",
    "kl": "kalau",
    "kalo": "kalau",
    "kalok": "kalau",

    "tp": "tapi",
    "tpi": "tapi",
    "tapii": "tapi",
    "namun": "tapi",

    "aja": "saja",
    "aj": "saja",
    "doang": "saja",

    "jg": "juga",
    "jga": "juga",
    "juga": "juga",

    "trs": "terus",
    "trus": "terus",
    "trusss": "terus",
    "teross": "terus",

    "sm": "sama",
    "sama": "sama",
    "ama": "sama",

    "pd": "pada",
    "pada": "pada",
    "di": "di",
    "ke": "ke",

    "dll": "danlainlain",
    "dsb": "dansebagainya",
    "dst": "danseterusnya",
    "dkk": "dankawankawan",

    "sbg": "sebagai",

    "bgt": "banget",
    "bgtt": "banget",
    "bngt": "banget",
    "bangettt": "banget",
    "bangett": "banget",
    "bgt.": "banget",

    "bener": "benar",
    "bner": "benar",
    "bner2": "benarbenar",
    "bener2": "benarbenar",
    "beneran": "benar",

    "pdhl": "padahal",
    "pdhal": "padahal",

    "slalu": "selalu",
    "selaluu": "selalu",
    "selaluuu": "selalu",

    "gitu": "begitu",
    "gituu": "begitu",
    "gini": "begini",
    "ginih": "begini",

    "nih": "ini",
    "ni": "ini",
    "inih": "ini",

    "yaa": "ya",
    "yaaa": "ya",
    "y": "ya",

    "iy": "iya",
    "iya": "iya",
    "iyaaa": "iya",

    "pake": "pakai",
    "dipake": "dipakai",
    "dipakein": "dipakai",
    "dipakaiin": "dipakai",
    "kepake": "terpakai",
    "kepakek": "terpakai",

    "bikin": "membuat",
    "bkin": "membuat",
    "bkn": "bukan",

    "dpt": "dapat",
    "dapet": "dapat",
    "dapat": "dapat",

    "krm": "kirim",
    "kirim": "kirim",

    "cek": "periksa",
    "check": "periksa",
    "ngecek": "memeriksa",
    "ngecheck": "memeriksa",

    "ngaruh": "berpengaruh",
    "ngaruhnya": "berpengaruh",
    "ngefek": "berpengaruh",
    "efek": "efek",

    "skincare": "skincare",
    "skin-care": "skincare",
    "skincaree": "skincare",

    "make-up": "makeup",
    "makeup": "makeup",

    "kosmetikk": "kosmetik",
    "kosmetik": "kosmetik",

    "product": "produk",
    "abal": "palsu",

    "skincarenya": "skincare",
    "makeupnya": "makeup",
    "produkny": "produknya",

    "mercury": "merkuri",
    "mercuri": "merkuri",

    "ijin": "izin",
    "ijinedar": "izinedar",

    "no": "nomor",
    "nomer": "nomor",

    "oknum2": "oknum",
    "produk2": "produk",
    "orang2": "orang",

    "gada": "tidak ada"
}

In [ ]:
def normalize_slang(text):
    text = text.lower()
    for slang, formal in kata.items():
        pattern = r"\b" + slang + r"\b"
        text = re.sub(pattern, formal, text)
    return text

databert["clean_text"] = databert["clean_text"].apply(normalize_slang)

In [ ]:
factory = StopWordRemoverFactory()
stopwords_id = factory.get_stop_words()

stopwords_custom = [
    "yang", "iya", "ya", "aja", "nih", "sih", "dong", "deh",
    "nya", "gak", "ga", "nggak", "enggak", "dan", "tapi",
    "inih", "rt", "via", "https", "http", "wkwk",
    "aku", "kamu", "dia", "mereka", "kita",
    "ini", "itu", "tsb", "dll"
]

stopwords_final = stopwords_id + stopwords_custom

In [ ]:
vectorizer = CountVectorizer(
    stop_words=stopwords_final,
    ngram_range=(1, 2),
    min_df=3
)

In [ ]:
topic_model = BERTopic(
    vectorizer_model=vectorizer,
    language="multilingual",
    calculate_probabilities=True,
    verbose=True)

In [ ]:
topics, probs = topic_model.fit_transform(
    databert["clean_text"].tolist())

2025-12-17 00:52:40,319 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

2025-12-17 00:53:03,925 - BERTopic - Embedding - Completed ✓
2025-12-17 00:53:03,928 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-12-17 00:53:05,525 - BERTopic - Dimensionality - Completed ✓
2025-12-17 00:53:05,526 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-12-17 00:53:05,563 - BERTopic - Cluster - Completed ✓
2025-12-17 00:53:05,568 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-17 00:53:05,600 - BERTopic - Representation - Completed ✓


In [ ]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,338,0_skincare_efek_pakai_samping,"[skincare, efek, pakai, samping, efek samping,...","[isinya skincare palsu palsu, skincare palsu p..."
1,1,184,1_kosmetik_berbahaya_bpom_kosmetik berbahaya,"[kosmetik, berbahaya, bpom, kosmetik berbahaya...",[bpom ungkap kosmetik berbahaya hai lagi dan l...
2,2,36,2_pakai_produk_bpom_baru,"[pakai, produk, bpom, baru, kemarin, termasuk,...",[keknya jangan pakai pinkflash dulu deh bes pr...
3,3,14,3_bahan_dampak_aman_hindari,"[bahan, dampak, aman, hindari, periksa, efek, ...",[tips aman pilih skincare yaitu periksa bahan ...


In [ ]:
topic_model.get_topic(0)

[('skincare', np.float64(0.2281041433703821)),
 ('efek', np.float64(0.09203820466374947)),
 ('pakai', np.float64(0.08858677198885886)),
 ('samping', np.float64(0.08765498607558893)),
 ('efek samping', np.float64(0.08680683203752697)),
 ('kalau', np.float64(0.08088270928309617)),
 ('banget', np.float64(0.06897626403097029)),
 ('kulit', np.float64(0.06714139657957392)),
 ('sama', np.float64(0.06585493720454953)),
 ('udah', np.float64(0.0615573207751212))]

In [ ]:
databert

,Text,Tanggal,Bulan,Tahun,sentimen,clean_text
0,enggak usah macem-macem lah kalau enggak paham...,16,11,2025,positif,tidak usah macem macem lah kalau tidak paham p...
1,Biar gada lagi oknum2 bikin kosmetik skincare ...,12,11,2025,netral,biar tidak ada lagi oknum membuat kosmetik ski...
2,Slalu update product makeup/skincare dari BPOM...,6,11,2025,netral,selalu update produk makeup skincare dari bpom...
3,untung deh gw belum beli skincare &amp; make u...,5,11,2025,netral,untung deh saya belum beli skincare amp make u...
4,Sebelum beli kosmetik pastikan Pilih kosmetik ...,28,10,2025,netral,sebelum beli kosmetik pastikan pilih kosmetik ...
...,...,...,...,...,...,...
567,ya ampun korban pink flash ngeri bgt untung ga...,7,11,2025,netral,ya ampun korban pink flash ngeri banget untung...
568,gila banyak bgt korbannya pink flash,7,11,2025,netral,gila banyak banget korbannya pink flash
569,@strsshdp @ywonnielvs Pink flash lagi di tarik...,7,11,2025,netral,pink flash lagi di tarik dari edaran kaa tiati ya
570,Pink flash blm BPOM jifrrr fak,7,11,2025,netral,pink flash blm bpom jifrrr fak


In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_list = [0,1,2,3]
databert['topic'] = topics
hasil_topik = databert[databert["topic"].isin(topic_list)][
    ["Text", "sentimen", "clean_text", "topic"]
]

hasil_topik.to_excel("hasil_bertopic_topik_0_3.xlsx", index=False)